# Step 3: assessment of the val runs → single test run
Reads the 6 runs saved by `13_real_training.ipynb` (`checkpoints/`); nothing is trained here. Shared code comes from `13_real_training.py`. The sections are the questions, in order: each one depends on the one before. Analysis cells are David's.

| § | Question | Outcome |
|---|---|---|
| 1 | Which pooling goes to test? | Tie within seed noise → **mean, seed 0** |
| 2 | Option B (siamese score + ZIP → LR): needed? | **No**: easy rejection 99.8–99.9% without the ZIP |
| 3 | Why did the 2026-10-21 run break? | **Unshuffled train loader** (David's sampler check) |
| 4 | Single test run, once, after §1–2 are decided → `results.md` | next |

In [1]:
%load_ext autoreload
%autoreload 2

import itertools
import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from helpers.pairs import load_pairs
from helpers.check_model_performance import review_typical_slices

build_vocab = import_module("siamese_model.01_vocab_step").build_vocab          # digit-prefixed filenames -> can't use `from ... import`
_data_step = import_module("siamese_model.11_data_plumbing")
PairDataset, make_collate = _data_step.PairDataset, _data_step.make_collate
predict = import_module("siamese_model.12_training_step").predict
_real_step = import_module("siamese_model.13_real_training")
evaluate, broken_run_problems = _real_step.evaluate, _real_step.broken_run_problems
load_run, rebuild_model = _real_step.load_run, _real_step.rebuild_model
THRESHOLD, CASE1_GUARDRAIL = _real_step.THRESHOLD, _real_step.CASE1_GUARDRAIL

POOLINGS = ["mean", "max"]
SEEDS = [0, 1, 2]
BATCH_SIZE = 64

In [ ]:
ot_train, ot_train_label = load_pairs("overture", "train")   # only for the vocab (train split only, same as 13)
ot_val, ot_val_label = load_pairs("overture", "val")

vocab = build_vocab(ot_train)
val_loader = DataLoader(PairDataset(ot_val, ot_val_label), batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(vocab))   # not shuffled: predict returns loader order

## 0. Load the runs
`summary`: one row per run (its best epoch's val numbers, epochs run, broken-run problems), from David's summary cell in 13. `histories[(pooling, seed)]`: every epoch of that run (`epoch, train_loss, val_loss, found, case1, easy`). Models: `rebuild_model(load_run(seed, pooling)[1])`.

In [3]:
summary_rows, histories = [], {}
for pool_mode, seed in itertools.product(POOLINGS, SEEDS):
    history, ckpt = load_run(seed, pool_mode)
    histories[(pool_mode, seed)] = history
    summary_rows.append({"pooling": pool_mode, "seed": seed, **(ckpt["best"] or {}), "epochs_run": len(history), "problems": broken_run_problems(history)})
summary = pd.DataFrame(summary_rows)
summary

,pooling,seed,epoch,train_loss,val_loss,found,case1,easy,epochs_run,problems
0,mean,0,10,0.003717,0.016915,87.076537,99.434003,99.932951,14,[]
1,mean,1,4,0.010259,0.014307,87.076537,99.499311,99.899426,8,[]
2,mean,2,5,0.008865,0.015387,90.840652,99.201800,99.878473,9,[]
3,max,0,9,0.004218,0.018681,88.958595,99.274363,99.895235,13,[]
4,max,1,6,0.007373,0.015520,88.205772,99.317901,99.886854,10,[]
5,max,2,5,0.008196,0.018080,90.213300,98.955083,99.832376,9,[]


## 1. Pooling choice
**Rule (agreed before computing):** a setting wins only if the gap between the two poolings' means is bigger than the seed spread (max − min over seeds) within each pooling; otherwise it's a tie. Tie on val found → lower val loss (the spec's epoch tie-break, extended); tie on that too → precedent.

In [ ]:
# Per pooling: mean over the 3 seeds
pooling_means = summary.groupby("pooling")[["train_loss", "val_loss", "found", "case1", "easy"]].mean()
display(pooling_means)
# Val loss: is the gap between poolings bigger than the seed spread within each pooling?


val_loss_gap = pooling_means.loc["mean", "val_loss"] - pooling_means.loc["max", "val_loss"]
val_loss_seed_spread = summary.groupby("pooling")["val_loss"].agg(lambda s: s.max() - s.min())
print(f"val loss gap (mean pooling − max pooling): {val_loss_gap:.4f}")
print("val loss seed spread (max − min over seeds):")
print(val_loss_seed_spread)

,train_loss,val_loss,found,case1,easy
pooling,,,,,
max,0.006595,0.017427,89.125889,99.182449,99.871489
mean,0.007613,0.015536,88.331242,99.378371,99.903616


val loss gap (mean pooling − max pooling): -0.0019
val loss seed spread (max − min over seeds):
pooling
max     0.003162
mean    0.002608
Name: val_loss, dtype: float64


**Observations (2026-10-23)**
- **Found: tie.** Max averages slightly higher, but the two poolings' seed ranges overlap (max's range sits inside mean's), and max wins only 2 of 3 seeds. Changing only the seed moves found more than changing the pooling does.
- **Val loss: tie.** Mean's lower average is a smaller gap than the seed spread within either pooling, so the seed alone could produce it.
- **Both poolings beat both baselines on val found in every run** (LogReg 81.8, RF 85.4), so neither choice leaves a better-than-baseline model behind.
- **Decision: mean pooling, seed 0.** Mean on precedent (Sentence-BERT, Neculoiu et al. both use mean pooling); seed 0 because it was the default before any run, not picked on val (picking the best-looking seed would send the luckiest run to test).
- **Caveats for the writeup.** (1) Each run's number is its best epoch, and found swings 2–4 pts between neighbouring epochs, so val numbers are slightly optimistic for both poolings. (2) The test number comes from one seed and carries that seed's luck: report it next to the val seed spread. (3) Finding: at 3 seeds, pooling (mean vs max) makes no detectable difference on this data.

## 2. Option B
Spec: put the ZIP back (siamese score + ZIP → LR) only if easy rejection falls behind without it. David's prediction: easy rejection < 98% without the ZIP.

In [5]:
chosen_run = summary[(summary["pooling"] == "mean") & (summary["seed"] == 0)]
chosen_run

,pooling,seed,epoch,train_loss,val_loss,found,case1,easy,epochs_run,problems
0,mean,0,10,0.003717,0.016915,87.076537,99.434003,99.932951,14,[]


**Observations (2026-10-23)**
- **Prediction wrong:** easy rejection is 99.8–99.9% in every run (chosen run 99.93) vs LogReg's 100.0 on val.
- **Why:** easy negatives are random pairs, almost always with a different street name, so the street text alone separates them. The ZIP mattered for LR because LR's street feature was weak (Step 1 street weight 0.6 vs ZIP 16); the siamese model reads the street characters directly.
- LR's 100% is not overfitting: it is measured on held-out val, and comes from the ZIP features (a different ZIP almost guarantees a non-match).
- **Caveat:** some address strings contain town + ZIP text (e.g. the UConn Foursquare string), so the model may not be entirely ZIP-free; the planned ZIP-text diagnostic (open questions) would size this.
- **Decision: no option B.** The gap is a fraction of a point on the easiest slice, while the siamese model is well ahead on found, the hard slice.

## 3. Why did the 2026-10-21 run break?
**Resolved (David, 2026-10-23):** an unshuffled train loader. David's own sampler check confirms the broken run's loader was sequential (it supersedes the `RandomSampler` reading logged on 2026-10-21). The train file is stored positives-first, so batches were long runs of one label. Consistent with the queue: `mean` / seed 0 (same seed and pooling, shuffled loader) trained normally from epoch 0 (found 78.7, val loss 0.024, the same as the first good run).

## 4. Single test run
Only after §1–2 are decided, and only once: `load_pairs("overture", "test")`, a non-shuffled loader, `rebuild_model` on the chosen checkpoint, `evaluate`. Then the `results.md` row + paragraph.

In [44]:
pooling_means = summary.groupby(["pooling", "seed"])[["train_loss", "val_loss", "found", "case1", "easy"]].mean()
display(pooling_means)
def _find_seed_noise(label):
    max_value = pooling_means.loc["mean", label].max()
    min_value = pooling_means.loc["mean", label].min()
    return max_value-min_value

for item in ["train_loss", "val_loss", "found","case1", "easy"]:
    print (f"diff in seed noise for column: {item}")
    print(_find_seed_noise(item))


train_loss  val_loss      found      case1       easy
pooling seed                                                       
max     0       0.004218  0.018681  88.958595  99.274363  99.895235
        1       0.007373  0.015520  88.205772  99.317901  99.886854
        2       0.008196  0.018080  90.213300  98.955083  99.832376
mean    0       0.003717  0.016915  87.076537  99.434003  99.932951
        1       0.010259  0.014307  87.076537  99.499311  99.899426
        2       0.008865  0.015387  90.840652  99.201800  99.878473

diff in seed noise for column: train_loss
0.006542080898575765
diff in seed noise for column: val_loss
0.0026076827198266983
diff in seed noise for column: found
3.7641154328732824
diff in seed noise for column: case1
0.2975110659603786
diff in seed noise for column: easy
0.05447764321334603


In [35]:
ot_test, ot_test_label = load_pairs("overture", "test")
test_loader = DataLoader(PairDataset(ot_test, ot_test_label), batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(vocab))   

ckpt = torch.load("checkpoints/overture_siamese_seed_0_pooling_mean.pt")

model = rebuild_model(ckpt)
# res = evaluate(model, test_loader, ot_test, ot_test_label)
# print(res)
val_preds = (predict(model, val_loader) >= THRESHOLD).int().numpy()
val_slices = review_typical_slices(ot_val, val_preds, ot_val_label)
print(val_slices)
preds = (predict(model, test_loader) >= THRESHOLD).int().numpy()
test_slices = review_typical_slices(ot_test, preds, ot_test_label)
print(test_slices)

                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3390       0    100.0%
1                  positives: non-identical    797      694     103     87.1%
2     neg: case1 (same street, diff number)  13781    13703      78     99.4%
3  neg: case2 (same number, similar street)     34       10      24     29.4%
4                         neg: case3 (easy)  23863    23847      16     99.9%
5                        overall (accuracy)  41865    41644     221     99.5%
                                      slice      n  correct  errors % correct
0                      positives: identical   3543     3543       0    100.0%
1                  positives: non-identical    843      705     138     83.6%
2     neg: case1 (same street, diff number)  14448    14367      81     99.4%
3  neg: case2 (same number, similar street)     15        5      10     33.3%
4                         neg: case3 (easy)  24995    24972     

**Baselines to compare against** (copied from `results.md`, Overture; test, val in brackets; % found for matches, % rejected for non-matches)

| Model | Step | Identical matches — % found | **Non-identical matches — % found** | Case1 — % rejected | Case2 — % rejected | Easy — % rejected |
|---|---|---|---|---|---|---|
| LogReg (3 Jaro-Winkler features) | 1 | 99.4 (99.5) | **79.2** (80.3) | 98.5 (98.3) | 6.7 (8.8) | 100.0 (100.0) |
| LogReg (23 similarity features) | 1b | 99.9 (100.0) | **79.0** (81.8) | 99.9 (99.8) | 6.7 (2.9) | 100.0 (100.0) |
| Random forest | 1b | 100.0 (100.0) | **83.0** (85.4) | 99.5 (99.5) | 6.7 (23.5) | 100.0 (100.0) |
| XGBoost | 1b | 100.0 (100.0) | **81.7** (84.9) | 99.7 (99.5) | 46.7 (20.6) | 99.9 (100.0) |
| libpostal, strict/strict/strict | 2c | 99.1 (99.0) | **52.3** (56.5) | 100.0 (100.0) | 80.0 (88.2) | 100.0 (100.0) |
| libpostal, loose/off/loose | 2c | 100.0 (100.0) | **71.9** (75.7) | 97.9 (98.3) | 73.3 (50.0) | 98.9 (99.4) |
| *test n* | | *3,543* | *843* | *14,448* | *15* | *24,995* |


## Diagnostics
One-off queries and their results (question → result → query). Kept out of the analysis cells.